# 感情AI: 第一段階(内部状態の浸透と滲み)— Kaggle 版(GPU: T4×2 / P100)

Colab の GPU 上限待ちの代替として Kaggle(週 30 時間、1 セッション 12 時間)を使う。
Colab 版 `Colab_run_c0.ipynb` と**同じスクリプト**(`run_c0.py`, `analyze_granularity.py`, …)を呼ぶだけで、コードは変えない。
違いは 3 点だけ: (1) Google Drive を使わず **`/kaggle/working`** に保存する、(2) リポジトリの置き場が `/kaggle/working/repo`、
(3) 関門 G1 用のセルがある(G1 のランナーは未実装。実装後にこのノートブックから呼ぶ)。

## 使い方(初回)

1. Kaggle にログインし、**電話番号の確認**(Settings → Phone verification)を済ませる。Internet と GPU はこれをしないと使えない。
2. 上のメニュー **Create → New Notebook**。右上の **File → Import Notebook** で、この `.ipynb` を GitHub の URL
   (`https://github.com/seina369/homeostatic-agent-experiments/blob/main/llm_grounding/Kaggle_run_stage1.ipynb`)
   か手元のファイルから読み込む。
3. 右側の **Settings** パネル(表示されていなければ右上の「<」または歯車アイコン)で:
   - **Accelerator**: `GPU T4 x2`(または `GPU P100`)。
   - **Internet**: **On** にする(GitHub の clone と Hugging Face のモデル取得に必要。電話番号の確認が済んでいないと切り替えられない)。
   - **Persistence**: `Files only` にしておくと、`/kaggle/working` の中身がセッションをまたいで残る(推奨)。
4. 上から順にセルを実行する(Ctrl+Enter)。

## 結果の残し方(重要)

- 対話セッションの `/kaggle/working` は、Persistence が `Files only` でないと**セッション終了で消える**。
- 確実に残すには、右上の **Save Version → Save & Run All (Commit)** でノートブック全体をバックグラウンド実行させる
  (最長 12 時間。ブラウザを閉じてもよい)。終わると `/kaggle/working` の中身が **Output** タブから
  ダウンロードできる(zip と SHA-256 の一覧は「8. 保存状況」のセルで作る)。
- 長い実行(C0 本番、G1)は Save & Run All で行い、対話セッションは動作確認と短い試走に使う。

## 進め方の順序(事前登録の 6 章)

このノートブックは **G1 まで**。G1 のランナー(GRPO を現行の系に繋ぐ)はまだ実装していないので、
「9. 関門 G1」のセルは呼び出し方の下書きだけ。実装後に同じセルから呼ぶ。


## 1. GPU 確認

Settings の Accelerator が `GPU T4 x2` または `GPU P100` になっていることを確認する(このセルが例外を出したら設定が違う)。

In [ ]:
!nvidia-smi
import torch
print("CUDA利用可能:", torch.cuda.is_available(), "| GPU数:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU{i}:", torch.cuda.get_device_name(i))
else:
    raise RuntimeError("GPUが見つからない。右側の Settings → Accelerator を「GPU T4 x2」か「GPU P100」にしてから、このセルからやり直すこと。")


## 2. GitHub から clone

`seina369/homeostatic-agent-experiments`(公開リポジトリ)を `/kaggle/working/repo` に置く。既にあれば `git pull`。
**Internet が Off だとここで失敗する**(Settings → Internet → On)。

In [ ]:
import os

REPO_URL = "https://github.com/seina369/homeostatic-agent-experiments.git"
REPO_DIR = "/kaggle/working/repo"
RESULTS_DIR = "/kaggle/working/results"          # Drive の MyDrive/EmotionalAI/ に相当

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.makedirs(RESULTS_DIR, exist_ok=True)
os.chdir(REPO_DIR)
LLM_DIR = os.path.join(REPO_DIR, "llm_grounding")
!ls {LLM_DIR}/run_c0.py {LLM_DIR}/seed_records.py {LLM_DIR}/torch_qwen_policy.py {LLM_DIR}/emotion_grounding_env.py {LLM_DIR}/grpo_trainer.py {LLM_DIR}/state_injector.py
!cd {REPO_DIR} && git log --oneline -1


## 3. 依存のインストールと環境の検算

`llm_grounding/requirements.txt`(torch は Kaggle 既存の CUDA 版をそのまま使う)。
Kaggle/Colab に同梱の `torchao` が peft の要求より古いと LoRA の付与で ImportError になる(追記欄「GRPO の実装」)ので外す。
最後に GPU 不要のテスト(GRPO・注入器・読み取り器)を回して、この環境で学習部分が動くことを確かめる。

In [ ]:
!pip install -q -r {LLM_DIR}/requirements.txt
!pip uninstall -y -q torchao 2>/dev/null || true
import transformers, peft, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| scikit-learn", sklearn.__version__)
!cd {LLM_DIR} && python3 -m pytest -q -p no:cacheprovider test_grpo.py test_state_injector.py test_leak_probe.py


## 4. 動作確認

環境定数が事前登録の確定値になっていること、モデルが読み込めて `respond()` が 1 回動くこと(Hugging Face からの取得に Internet が必要)。

In [ ]:
import sys
sys.path.insert(0, LLM_DIR)
import emotion_grounding_env as E
from seed_records import env_constants
from torch_qwen_policy import TorchPolicy

print("環境定数:", env_constants())
assert (E.B0, E.BUDGET_LOW_THRESHOLD, E.U_OPT, E.U_MIN, E.U_MAX) == (340, 85.0, 0.7, 0.0, 2.5), "事前登録の確定値と違う"
assert E.PROMPT_VERSION == 2, "v2 のプロンプトになっていない(git pull を確認)"
print(E.PROMPT_TEMPLATE)

_p = TorchPolicy(verbose=False)
_r = _p.respond(E.PROMPT_TEMPLATE.format(task="What is 23 + 19?", budget=340, error=0, uncertainty=0.70))
print(repr(_r.text[:120]), f"n_tokens={_r.n_tokens} mean_entropy={_r.mean_entropy:.3f}")
del _p, _r
torch.cuda.empty_cache()


## 5. 速度の実測(任意)

Colab の T4 での実測(12.47 秒/episode)と比べるため。P100 や T4×2(このコードは 1 枚しか使わない)で
所要時間が変わるかを見る。1 seed × 3 episode。結果は本番の標本に含めない。

In [ ]:
import time
SPEED_DIR = f"{RESULTS_DIR}/speed_check"
t0 = time.time()
!python3 {LLM_DIR}/run_c0.py --policy torch --seeds 1 --episodes 3 --temperature 1.0 --out-dir "{SPEED_DIR}" --force
print(f"所要時間: {time.time() - t0:.0f}秒(モデル読み込み込み)")


## 6. C0 の実行(Colab 版の「6. 実行」と同じ)

`run_c0.py` を同じ引数で呼ぶ。保存先は `/kaggle/working/results/c0_v2/`。途中で切れたら、このセルをもう一度実行する
(完了済み seed は飛ばされる)。seed 数・episode 数は事前登録の確定値(15 × 30。episode 数は 7 章のとおり G1 後に 30 か 45 に確定)。
長い実行は Save & Run All で行うこと。

In [ ]:
import time
C0_DIR = f"{RESULTS_DIR}/c0_v2"
SEEDS, EPISODES = 15, 30
os.makedirs(C0_DIR, exist_ok=True)
t0 = time.time()
!python3 {LLM_DIR}/run_c0.py --policy torch --seeds {SEEDS} --episodes {EPISODES} --temperature 1.0 --out-dir "{C0_DIR}"
print(f"所要時間: {(time.time() - t0) / 60:.1f}分")


## 7. 分析(C0 完了後。分析コードは変更しない)

`analyze_granularity.py`(v1 の主指標)と `leak_probe.py`(第一段階の読み取り器: 5 分割、行の並べ替え帰無 100 回)。
結果の JSON も `/kaggle/working/results` に置く。

In [ ]:
!python3 {LLM_DIR}/analyze_granularity.py --dir "{C0_DIR}" --out "{C0_DIR}/c0_analysis.json"
!python3 {LLM_DIR}/leak_probe.py --dir "{C0_DIR}" --out "{C0_DIR}/c0_leak_probe.json"


## 8. 保存状況の確認と取り出し

seed ごとのファイルが揃っているかを見て、`/kaggle/working/results` を zip にまとめ、SHA-256 の一覧を出す
(手元に取り込むときの照合用。Colab 版と同じ手順)。zip は Output タブ(Save Version 後)か、対話セッションなら
右側の Output パネルからダウンロードする。

In [ ]:
import json, glob, hashlib, zipfile
files = sorted(glob.glob(f"{C0_DIR}/c0_seed*.json"))
print(f"{len(files)} / {SEEDS} seed 保存済み({C0_DIR})")
for f in files:
    with open(f, encoding="utf-8") as fh:
        p = json.load(fh)
    print(f"  {os.path.basename(f)}: complete={p.get('complete')} episodes={p.get('n_episodes')} "
          f"steps={p.get('n_steps')} elapsed={p.get('elapsed_seconds', 0) / 60:.1f}min model={p.get('policy', {}).get('model')}")

zip_path = "/kaggle/working/results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, names in os.walk(RESULTS_DIR):
        for n in names:
            full = os.path.join(root, n)
            z.write(full, arcname=os.path.relpath(full, "/kaggle/working"))
            print(os.path.relpath(full, "/kaggle/working"), os.path.getsize(full), hashlib.sha256(open(full, "rb").read()).hexdigest())
print("ZIP", os.path.getsize(zip_path), hashlib.sha256(open(zip_path, "rb").read()).hexdigest())


## 9. 関門 G1: GRPO を現行の系に繋ぐ(事前登録 6 章)

**G1 のランナーは未実装**(このノートブックの時点では `grpo_core.py` / `grpo_trainer.py` / `state_injector.py` と
その CPU テストまで)。実装して commit したら、このセルの呼び出しを有効にして Save & Run All で回す。
G1 の合格条件: 3 seed で学習後の平均逸脱が学習前より 5% 以上下がる、損失・報酬が NaN にならない、
1 seed の学習+評価が 90 分に収まる。ここで G・学習率・KL 係数・更新回数を固定する。

In [ ]:
G1_DIR = f"{RESULTS_DIR}/g1"
os.makedirs(G1_DIR, exist_ok=True)
G1_RUNNER = os.path.join(LLM_DIR, "run_g1.py")   # 未実装。実装後に同名で置く
if os.path.exists(G1_RUNNER):
    import time
    t0 = time.time()
    !python3 {G1_RUNNER} --seeds 3 --out-dir "{G1_DIR}"
    print(f"所要時間: {(time.time() - t0) / 60:.1f}分")
else:
    print("run_g1.py はまだ実装されていない(事前登録 6 章 G1 の設計に従って実装してから、このセルを実行する)。")
